In [1]:
import numpy as np
import pandas as pd


In [2]:
data = pd.read_csv("/Users/nidhishgupta/Desktop/Dream11_Fantasy Team_Predictor/data/processed/ready_for_training.csv")

In [3]:
data

,match_id,player,fantasy_points,venue,season,player_match_number,last3_avg_points,last5_avg_points,last10_avg_points,rolling_strike_rate,...,stage_Final,stage_Qualifier 1,stage_Qualifier 2,stage_Semi Final,stage_Unknown,pitch_type_batting_friendly,pitch_type_pace_friendly,pitch_type_spin_friendly,player_role_batsman,player_role_bowler
0,548341,0,58.0,53,2012,1,35.672040,35.67204,35.67204,80.762144,...,False,False,False,False,True,False,False,False,False,True
1,548346,0,41.0,56,2012,2,35.672040,35.67204,35.67204,80.762144,...,False,False,False,False,True,True,False,False,False,False
2,548348,0,23.0,2,2012,3,35.672040,35.67204,35.67204,80.762144,...,False,False,False,False,True,False,False,False,False,True
3,548352,0,28.0,27,2012,4,40.666667,35.67204,35.67204,80.762144,...,False,False,False,False,True,False,False,True,False,True
4,548356,0,33.0,23,2012,5,30.666667,35.67204,35.67204,80.762144,...,False,False,False,False,True,True,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25120,1473478,766,25.0,44,2025,6,3.333333,25.60000,35.67204,0.000000,...,False,False,False,False,True,False,False,False,False,True
25121,1473480,766,0.0,28,2025,7,11.666667,13.60000,35.67204,0.000000,...,False,False,False,False,True,False,False,True,False,False
25122,1473488,766,23.0,33,2025,8,12.333333,7.00000,35.67204,0.000000,...,False,False,False,False,True,False,False,False,False,True
25123,1473492,766,4.0,44,2025,9,16.000000,11.60000,35.67204,0.000000,...,False,False,False,False,True,False,False,False,False,True


In [4]:
data.columns

Index(['match_id', 'player', 'fantasy_points', 'venue', 'season',
       'player_match_number', 'last3_avg_points', 'last5_avg_points',
       'last10_avg_points', 'rolling_strike_rate', 'rolling_wickets',
       'venue_avg_points', 'opponent_avg_points', 'venue_run_factor',
       'batting_contribution_ratio', 'bowling_contribution_ratio',
       'last10_std_points', 'player_consistency_index', 'form_momentum',
       'bat_pos', 'boundary_percentage', 'bat_pos_per_match',
       'player_role_wicketkeeper', 'match_month', 'team_won_toss',
       'team_Deccan Chargers', 'team_Delhi Capitals', 'team_Gujarat Lions',
       'team_Gujarat Titans', 'team_Kochi Tuskers Kerala',
       'team_Kolkata Knight Riders', 'team_Lucknow Super Giants',
       'team_Mumbai Indians', 'team_Pune Warriors', 'team_Punjab Kings',
       'team_Rajasthan Royals', 'team_Rising Pune Supergiants',
       'team_Royal Challengers Bengaluru', 'team_Sunrisers Hyderabad',
       'opponent_Deccan Chargers', 'opponent_D

In [5]:
data['season'] = data['season'].str[:4].astype(int)
data['season'].unique()

array([2012, 2013, 2015, 2016, 2022, 2023, 2024, 2025, 2007, 2009, 2017,
       2018, 2011, 2014, 2019, 2020, 2021])

In [84]:
train = data[data['season'] <= 2022]
test  = data[data['season'] > 2022]
X_train = train.drop(columns=['fantasy_points','player','venue','season'])
y_train = train['fantasy_points']

X_test = test.drop(columns=['fantasy_points','player','venue','season'])
y_test = test['fantasy_points']

In [85]:
from xgboost import XGBRegressor

xg = XGBRegressor(
    n_estimators=900,
    learning_rate=0.03,
    max_depth=7,
    subsample=0.85,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    random_state=42
)

xg.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [86]:
y_pred = xg.predict(X_test)

In [87]:
from sklearn.metrics import mean_absolute_error
print(mean_absolute_error(y_test, y_pred))

14.111186512407462


In [88]:

importance = pd.Series(
    xg.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importance.head(20))

player_role_bowler               0.224609
player_role_batsman              0.143719
venue_avg_points                 0.084186
bowling_contribution_ratio       0.058492
boundary_percentage              0.041484
batting_contribution_ratio       0.027691
opponent_avg_points              0.023643
bat_pos_per_match                0.013721
stage_Qualifier 2                0.010076
rolling_wickets                  0.009515
opponent_Sunrisers Hyderabad     0.009180
bat_pos                          0.008743
opponent_Rajasthan Royals        0.008733
opponent_Lucknow Super Giants    0.008587
stage_Unknown                    0.008533
last10_std_points                0.008413
player_role_wicketkeeper         0.008293
team_Lucknow Super Giants        0.008153
last10_avg_points                0.008147
rolling_strike_rate              0.008128
dtype: float32


In [89]:
X_train

,player_match_number,last3_avg_points,last5_avg_points,last10_avg_points,rolling_strike_rate,rolling_wickets,venue_avg_points,opponent_avg_points,venue_run_factor,batting_contribution_ratio,...,stage_Final,stage_Qualifier 1,stage_Qualifier 2,stage_Semi Final,stage_Unknown,pitch_type_batting_friendly,pitch_type_pace_friendly,pitch_type_spin_friendly,player_role_batsman,player_role_bowler
0,1,35.672040,35.67204,35.67204,80.762144,0.0,58.00,39.250000,0.879923,0.000000,...,False,False,False,False,True,False,False,False,False,True
1,2,35.672040,35.67204,35.67204,80.762144,0.0,41.00,30.000000,1.010270,0.292683,...,False,False,False,False,True,True,False,False,False,False
2,3,35.672040,35.67204,35.67204,80.762144,0.0,23.00,39.250000,1.027630,0.000000,...,False,False,False,False,True,False,False,False,False,True
3,4,40.666667,35.67204,35.67204,80.762144,0.0,45.00,47.333333,0.988070,0.107143,...,False,False,False,False,True,False,False,True,False,True
4,5,30.666667,35.67204,35.67204,80.762144,0.0,31.75,38.714286,0.964953,0.000000,...,False,False,False,False,True,True,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25110,95,19.333333,29.40000,25.80000,0.000000,1.0,11.00,39.000000,0.955192,0.000000,...,False,False,False,False,True,False,False,False,False,False
25111,96,0.000000,13.20000,22.70000,0.000000,0.4,38.00,26.600000,0.970209,1.000000,...,False,False,False,False,True,True,False,False,True,False
25112,97,-1.333333,10.80000,22.10000,3.636364,0.4,8.00,13.000000,1.012649,0.000000,...,False,False,False,False,True,False,False,True,False,False
25113,98,1.333333,0.80000,22.70000,3.636364,0.0,38.00,70.500000,0.970209,0.000000,...,False,False,False,False,True,True,False,False,False,True


In [90]:
X_train = X_train.drop(columns=[
    "batting_contribution_ratio",
    "bowling_contribution_ratio",
    "bat_pos_per_match",
    "stage_Final",
    "stage_Qualifier 2",
    "stage_Eliminator",
    "stage_Semi Final",
    "stage_Unknown"
])
X_test = X_test.drop(columns=[
    "batting_contribution_ratio",
    "bowling_contribution_ratio",
    "bat_pos_per_match",
    "stage_Final",
    "stage_Qualifier 2",
    "stage_Eliminator",
    "stage_Semi Final",
    "stage_Unknown"
])
X_train['recent_form'] = (X_train['last3_avg_points']*0.6+X_train['last5_avg_points']*0.4+X_train['last10_avg_points']*0.1)
X_test['recent_form'] = (X_test['last3_avg_points']*0.6+X_test['last5_avg_points']*0.4+X_test['last10_avg_points']*0.1)
X_train['venue_form'] = X_train['venue_avg_points']*X_train['recent_form'];
X_test['venue_form'] = X_test['venue_avg_points']*X_test['recent_form'];

In [92]:
X_train.columns

Index(['player_match_number', 'last3_avg_points', 'last5_avg_points',
       'last10_avg_points', 'rolling_strike_rate', 'rolling_wickets',
       'venue_avg_points', 'opponent_avg_points', 'venue_run_factor',
       'last10_std_points', 'player_consistency_index', 'form_momentum',
       'bat_pos', 'boundary_percentage', 'player_role_wicketkeeper',
       'match_month', 'team_won_toss', 'team_Deccan Chargers',
       'team_Delhi Capitals', 'team_Gujarat Lions', 'team_Gujarat Titans',
       'team_Kochi Tuskers Kerala', 'team_Kolkata Knight Riders',
       'team_Lucknow Super Giants', 'team_Mumbai Indians',
       'team_Pune Warriors', 'team_Punjab Kings', 'team_Rajasthan Royals',
       'team_Rising Pune Supergiants', 'team_Royal Challengers Bengaluru',
       'team_Sunrisers Hyderabad', 'opponent_Deccan Chargers',
       'opponent_Delhi Capitals', 'opponent_Gujarat Lions',
       'opponent_Gujarat Titans', 'opponent_Kochi Tuskers Kerala',
       'opponent_Kolkata Knight Riders', 'opp

In [95]:
model = XGBRegressor(
    n_estimators=900,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=4,
    subsample=0.85,
    colsample_bytree=0.6,
    reg_alpha=0.7,
    reg_lambda=2,
    random_state=42
)
model.fit(X_train,y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.6
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [97]:
y_pred = model.predict(X_test)
print(mean_absolute_error(y_test, y_pred))
importance = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importance.head(20))

14.458035451253956
player_role_bowler                0.181259
venue_avg_points                  0.173571
player_role_batsman               0.097132
opponent_avg_points               0.066731
boundary_percentage               0.058842
venue_form                        0.044150
bat_pos                           0.028579
recent_form                       0.015356
rolling_wickets                   0.013671
last10_avg_points                 0.010251
last3_avg_points                  0.010138
last10_std_points                 0.009868
opponent_Sunrisers Hyderabad      0.009553
rolling_strike_rate               0.009165
pitch_type_pace_friendly          0.009113
opponent_Deccan Chargers          0.008796
last5_avg_points                  0.008695
player_match_number               0.008520
team_Pune Warriors                0.008230
opponent_Kolkata Knight Riders    0.008201
dtype: float32


In [100]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)



from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor
import numpy as np
param_dist = {
    "max_depth": [3,4,5,6],
    "learning_rate": np.linspace(0.01,0.05,5),
    "n_estimators": [500,700,900,1100],
    "min_child_weight": [2,3,4,5],
    "subsample": [0.75,0.8,0.85],
    "colsample_bytree": [0.5,0.6,0.7],
    
    # regularization
    "reg_alpha": [0,0.3,0.5,1],
    "reg_lambda": [1,1.5,2,3],
    "gamma": [0,0.05,0.1,0.2]
}
model = XGBRegressor(
    objective="reg:squarederror",
    random_state=42
)
random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=60,
    scoring="neg_mean_absolute_error",
    cv=tscv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

Fitting 5 folds for each of 60 candidates, totalling 300 fits


,estimator,"XGBRegressor(...state=42, ...)"
,param_distributions,"{'colsample_bytree': [0.5, 0.6, ...], 'gamma': [0, 0.05, ...], 'learning_rate': array([0.01, ..., 0.04, 0.05]), 'max_depth': [3, 4, ...], ...}"
,n_iter,60
,scoring,'neg_mean_absolute_error'
,n_jobs,-1
,refit,True
,cv,TimeSeriesSpl...est_size=None)
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [101]:
print("Best parameters:", random_search.best_params_)
best_mae = -random_search.best_score_
print("Best MAE:", best_mae)

Best parameters: {'subsample': 0.85, 'reg_lambda': 1, 'reg_alpha': 0.3, 'n_estimators': 900, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': np.float64(0.01), 'gamma': 0.2, 'colsample_bytree': 0.6}
Best MAE: 14.908980668438975


In [102]:
param_grid = {
    "max_depth": [3,4,5],
    "learning_rate": [0.03,0.04],
    "n_estimators": [700,900],
    "min_child_weight": [3,4],
    "subsample": [0.8,0.85],
    "colsample_bytree": [0.6,0.7],
    "reg_alpha": [0.5,1],
    "reg_lambda": [1,2],
    "gamma": [0,0.1]
}
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=42
)
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=5,
    verbose=2,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 768 candidates, totalling 3840 fits
[CV] END colsample_bytree=0.6, gamma=0.2, learning_rate=0.05, max_depth=5, min_child_weight=5, n_estimators=500, reg_alpha=1, reg_lambda=1, subsample=0.85; total time=   1.1s
[CV] END colsample_bytree=0.7, gamma=0.05, learning_rate=0.03, max_depth=6, min_child_weight=2, n_estimators=1100, reg_alpha=0.5, reg_lambda=3, subsample=0.85; total time=   3.6s
[CV] END colsample_bytree=0.6, gamma=0.2, learning_rate=0.05, max_depth=3, min_child_weight=2, n_estimators=700, reg_alpha=0.3, reg_lambda=3, subsample=0.85; total time=   1.1s
[CV] END colsample_bytree=0.5, gamma=0.2, learning_rate=0.03, max_depth=6, min_child_weight=3, n_estimators=1100, reg_alpha=0.3, reg_lambda=3, subsample=0.8; total time=   3.2s
[CV] END colsample_bytree=0.7, gamma=0, learning_rate=0.05, max_depth=5, min_child_weight=5, n_estimators=700, reg_alpha=0, reg_lambda=3, subsample=0.75; total time=   1.7s
[CV] END colsample_bytree=0.6, gamma=0.05, learning_rat

,estimator,"XGBRegressor(...state=42, ...)"
,param_grid,"{'colsample_bytree': [0.6, 0.7], 'gamma': [0, 0.1], 'learning_rate': [0.03, 0.04], 'max_depth': [3, 4, ...], ...}"
,scoring,'neg_mean_absolute_error'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'reg:squarederror'


In [103]:
best_model = grid_search.best_estimator_
preds = best_model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
print("MAE:", mae)
importance = pd.Series(
    best_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importance.head(20))

MAE: 14.482194558209045
player_role_bowler                  0.169416
venue_avg_points                    0.149538
player_role_batsman                 0.078307
boundary_percentage                 0.059374
opponent_avg_points                 0.053596
venue_form                          0.037640
bat_pos                             0.024100
recent_form                         0.019220
rolling_wickets                     0.013751
last3_avg_points                    0.012033
opponent_Sunrisers Hyderabad        0.011411
stage_Qualifier 1                   0.011284
team_Royal Challengers Bengaluru    0.011223
last10_std_points                   0.010956
last10_avg_points                   0.010920
pitch_type_pace_friendly            0.010573
rolling_strike_rate                 0.010538
opponent_Rajasthan Royals           0.010110
last5_avg_points                    0.010098
opponent_Punjab Kings               0.009792
dtype: float32


In [42]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
xgb = XGBRegressor(
    objective='reg:squarederror',
    random_state=42
)
param_grid = {
    'n_estimators': [300, 500, 700],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.03, 0.05],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8],
    'min_child_weight': [1, 3, 5]
}
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=5,
    verbose=2,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 486 candidates, totalling 2430 fits
[CV] END colsample_bytree=0.7, learning_rate=0.01, max_depth=4, min_child_weight=1, n_estimators=300, subsample=0.7; total time=   0.7s
[CV] END colsample_bytree=0.7, learning_rate=0.01, max_depth=4, min_child_weight=1, n_estimators=300, subsample=0.9; total time=   0.7s
[CV] END colsample_bytree=0.7, learning_rate=0.01, max_depth=4, min_child_weight=1, n_estimators=500, subsample=0.8; total time=   1.0s
[CV] END colsample_bytree=0.7, learning_rate=0.01, max_depth=4, min_child_weight=1, n_estimators=500, subsample=0.9; total time=   1.1s
[CV] END colsample_bytree=0.7, learning_rate=0.01, max_depth=4, min_child_weight=1, n_estimators=700, subsample=0.8; total time=   1.4s
[CV] END colsample_bytree=0.7, learning_rate=0.01, max_depth=4, min_child_weight=3, n_estimators=300, subsample=0.7; total time=   0.7s
[CV] END colsample_bytree=0.7, learning_rate=0.01, max_depth=4, min_child_weight=3, n_estimators=300, subsample=0.7; tot

,estimator,"XGBRegressor(...state=42, ...)"
,param_grid,"{'colsample_bytree': [0.7, 0.8], 'learning_rate': [0.01, 0.03, ...], 'max_depth': [4, 6, ...], 'min_child_weight': [1, 3, ...], ...}"
,scoring,'neg_mean_absolute_error'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'reg:squarederror'


In [43]:
best_model = grid_search.best_estimator_

In [44]:
best_model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.7
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [45]:
from sklearn.metrics import mean_absolute_error

preds = best_model.predict(X_test)

mae = mean_absolute_error(y_test, preds)

print("MAE:", mae)

MAE: 13.663024602707539


In [46]:
importance = pd.Series(
    best_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importance.head(20))

player_role_bowler                      0.200347
venue_avg_points                        0.145733
bowling_contribution_ratio              0.068818
boundary_percentage                     0.059921
opponent_avg_points                     0.046107
player_role_batsman                     0.041079
batting_contribution_ratio              0.032373
bat_pos_per_match                       0.020278
bat_pos                                 0.012910
rolling_wickets                         0.011361
opponent_Sunrisers Hyderabad            0.011198
team_Royal Challengers Bengaluru        0.010862
opponent_Rajasthan Royals               0.010689
stage_Final                             0.010508
opponent_Royal Challengers Bengaluru    0.010251
opponent_Lucknow Super Giants           0.010163
team_Punjab Kings                       0.010020
last10_std_points                       0.009992
last10_avg_points                       0.009905
stage_Qualifier 2                       0.009723
dtype: float32


In [48]:
X_train.columns

Index(['player_match_number', 'last3_avg_points', 'last5_avg_points',
       'last10_avg_points', 'rolling_strike_rate', 'rolling_wickets',
       'venue_avg_points', 'opponent_avg_points', 'venue_run_factor',
       'batting_contribution_ratio', 'bowling_contribution_ratio',
       'last10_std_points', 'player_consistency_index', 'form_momentum',
       'bat_pos', 'boundary_percentage', 'bat_pos_per_match',
       'player_role_wicketkeeper', 'match_month', 'team_won_toss',
       'team_Delhi Capitals', 'team_Gujarat Lions', 'team_Gujarat Titans',
       'team_Kolkata Knight Riders', 'team_Lucknow Super Giants',
       'team_Mumbai Indians', 'team_Punjab Kings', 'team_Rajasthan Royals',
       'team_Royal Challengers Bengaluru', 'team_Sunrisers Hyderabad',
       'opponent_Delhi Capitals', 'opponent_Gujarat Lions',
       'opponent_Gujarat Titans', 'opponent_Kolkata Knight Riders',
       'opponent_Lucknow Super Giants', 'opponent_Mumbai Indians',
       'opponent_Punjab Kings', 'oppon